# Coffee Sales Analysis with PySpark
## Big Data Processing and Analysis

This notebook demonstrates Big Data processing using Apache Spark.

### Contents:
1. Spark Session Setup
2. Data Loading
3. Data Exploration
4. Aggregations and Analytics
5. Advanced Transformations
6. Performance Optimization

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, max as spark_max, min as spark_min,
    year, month, dayofmonth, hour, dayofweek, date_format,
    to_date, to_timestamp, expr, lit, when, round as spark_round
)
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from pathlib import Path

## 1. Spark Session Setup

In [ ]:
# Create Spark session
spark = SparkSession.builder \
    .appName("CoffeeSalesAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")

## 2. Data Loading

In [ ]:
# Load data
data_path = Path('../data/cleaned/')

# Try to load parquet first
parquet_files = list(data_path.glob('*.parquet'))
if parquet_files:
    latest_file = str(max(parquet_files, key=lambda x: x.stat().st_mtime))
    df = spark.read.parquet(latest_file)
    print(f"Loaded data from: {Path(latest_file).name}")
else:
    csv_files = list(data_path.glob('*.csv'))
    latest_file = str(max(csv_files, key=lambda x: x.stat().st_mtime))
    df = spark.read.csv(latest_file, header=True, inferSchema=True)
    print(f"Loaded data from: {Path(latest_file).name}")

print(f"\nNumber of rows: {df.count():,}")
print(f"Number of columns: {len(df.columns)}")

## 3. Data Exploration

In [ ]:
# Display schema
print("Schema:")
df.printSchema()

In [ ]:
# Show first few rows
print("First 10 rows:")
df.show(10, truncate=False)

In [ ]:
# Basic statistics
print("Basic Statistics:")
df.describe().show()

In [ ]:
# Check for nulls
print("Null counts per column:")
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show()

## 4. Aggregations and Analytics

In [ ]:
# Overall aggregations
numeric_cols = [field.name for field in df.schema.fields 
                if str(field.dataType) in ['IntegerType', 'DoubleType', 'FloatType', 'LongType']]

if numeric_cols:
    print("Overall Aggregations:")
    
    # Create aggregation expressions
    agg_exprs = []
    for col_name in numeric_cols[:3]:  # First 3 numeric columns
        agg_exprs.extend([
            spark_sum(col_name).alias(f"{col_name}_sum"),
            avg(col_name).alias(f"{col_name}_avg"),
            spark_max(col_name).alias(f"{col_name}_max"),
            spark_min(col_name).alias(f"{col_name}_min")
        ])
    
    df.agg(*agg_exprs).show()

In [ ]:
# Group by categorical columns
categorical_cols = [field.name for field in df.schema.fields 
                    if str(field.dataType) == 'StringType']

if categorical_cols:
    cat_col = categorical_cols[0]
    print(f"\nGroup by {cat_col}:")
    
    df.groupBy(cat_col) \
      .agg(count("*").alias("count")) \
      .orderBy(col("count").desc()) \
      .show(10)

## 5. Time Series Analysis

In [ ]:
# Find date columns
date_cols = [field.name for field in df.schema.fields 
             if 'date' in field.name.lower() or 'time' in field.name.lower()]

if date_cols:
    date_col = date_cols[0]
    print(f"Time series analysis using: {date_col}")
    
    # Convert to timestamp
    df_time = df.withColumn("timestamp", to_timestamp(col(date_col)))
    
    # Extract time components
    df_time = df_time.withColumn("year", year("timestamp")) \
                     .withColumn("month", month("timestamp")) \
                     .withColumn("day", dayofmonth("timestamp")) \
                     .withColumn("hour", hour("timestamp")) \
                     .withColumn("dayofweek", dayofweek("timestamp"))
    
    # Hourly pattern
    print("\nTransactions by Hour:")
    df_time.groupBy("hour") \
           .agg(count("*").alias("transaction_count")) \
           .orderBy("hour") \
           .show()
    
    # Day of week pattern
    print("\nTransactions by Day of Week:")
    df_time.groupBy("dayofweek") \
           .agg(count("*").alias("transaction_count")) \
           .orderBy("dayofweek") \
           .show()

## 6. Window Functions

In [ ]:
# Window functions for ranking and analytics
if categorical_cols and numeric_cols:
    cat_col = categorical_cols[0]
    num_col = numeric_cols[0]
    
    print(f"Window function: Ranking by {num_col} within {cat_col}")
    
    # Define window
    window_spec = Window.partitionBy(cat_col).orderBy(col(num_col).desc())
    
    # Add rank
    df_ranked = df.withColumn("rank", F.rank().over(window_spec))
    
    # Show top 3 per category
    df_ranked.filter(col("rank") <= 3) \
             .select(cat_col, num_col, "rank") \
             .orderBy(cat_col, "rank") \
             .show(20)

## 7. Advanced Transformations

In [ ]:
# Create derived features
if numeric_cols and len(numeric_cols) >= 2:
    col1, col2 = numeric_cols[0], numeric_cols[1]
    
    print(f"Creating derived features from {col1} and {col2}")
    
    df_derived = df.withColumn(
        "ratio",
        when(col(col2) != 0, col(col1) / col(col2)).otherwise(0)
    )
    
    df_derived.select(col1, col2, "ratio").show(10)

## 8. Save Results

In [ ]:
# Save processed data
output_path = '../data/processed/spark_notebook_output'

# Save as parquet
df.write.mode('overwrite').parquet(output_path)
print(f"Saved processed data to: {output_path}")

In [ ]:
# Stop Spark session
spark.stop()
print("Spark session stopped")

## Summary

This notebook demonstrated:
- Setting up a Spark session for big data processing
- Loading and exploring data with Spark
- Performing aggregations and analytics
- Time series analysis
- Using window functions
- Creating derived features
- Saving processed results

Spark enables processing of large datasets that don't fit in memory, with distributed computing capabilities.